In [ ]:
# ==========================================
# STUDENT FINAL EXAM EARLY WARNING SYSTEM
# 1-MONTH BEFORE EXAM (RULE + ML)
# ==========================================

# ---------- 1. IMPORTS ----------
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output
import sys
import uuid
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

# ---------- 2. LOAD DATA ----------
df = pd.read_csv("Team_wizards_balanced.csv")

df = df.rename(columns={
    "Attendance (%)": "AttendanceRate",
    "Study Hours": "StudyHoursPerWeek"
})

df["FinalGrade"] = df["FinalGrade"].map({"Pass": 1, "Fail": 0})

FEATURES = ["AttendanceRate", "StudyHoursPerWeek", "PreviousGrade"]
TARGET = "FinalGrade"

df = df.dropna(subset=FEATURES + [TARGET])

# ---------- CHECK CLASS BALANCE ----------
print("\n--- CLASS DISTRIBUTION ---")
print(df[TARGET].value_counts())
print(f"Pass: {df[TARGET].sum()} | Fail: {(df[TARGET]==0).sum()}")

X = df[FEATURES]
y = df[TARGET]

# ---------- 3. TRAIN TEST SPLIT ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------- 4. TRAIN MODEL ----------
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=5,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

# ---------- 5. MODEL EVALUATION ----------
print("\n--- MODEL PERFORMANCE ---")
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))


# ==========================================
# STUDENT SELF-CHECK (1 MONTH BEFORE EXAM)
# ==========================================

def get_valid_input(prompt, min_val, max_val):
    while True:
        try:
            v = float(input(prompt))
            if min_val <= v <= max_val:
                return v
            print(f"❌ Enter value between {min_val} and {max_val}")
        except:
            print("❌ Invalid number")

def get_reg_no():
    while True:
        reg = input("Enter register number (12 digits): ").strip()
        if reg.isdigit() and len(reg) == 12 and reg.startswith("7142"):
            clear_output(wait=True)
            return reg
        print("❌ Invalid register number. Must be 12 digits and start with 7142.")

def get_academic_year():
    while True:
        year = input("Enter academic year (YYYY-YYYY): ").strip()
        if (
            len(year) == 9 and
            year[4] == "-" and
            year[:4].isdigit() and
            year[5:].isdigit() and
            int(year[5:]) == int(year[:4]) + 1
        ):
            clear_output(wait=True)
            return year
        print("❌ Invalid academic year. Example: 2025-2026")

def get_subject_code():
    while True:
        code = input("Enter subject code: ").strip().upper()
        if len(code) == 6 and code[:2].isalpha() and code[2:].isdigit():
            return code
        print("❌ Invalid subject code. Format must be like AI3391.")

def get_subject_difficulty():
    while True:
        diff = input("Enter subject difficulty (easy/moderate/hard): ").strip().lower()
        if diff in ["easy", "moderate", "hard"]:
            clear_output(wait=True)
            return diff
        print("❌ Invalid difficulty. Choose easy, moderate, or hard.")

print("\n--- STUDENT FINAL EXAM SELF-CHECK ---")

reg_no = get_reg_no()
year = get_academic_year()
subject_code = get_subject_code()
difficulty = get_subject_difficulty()

study_hours = get_valid_input("Daily study hours (0–12): ", 0, 12)
attendance = get_valid_input("Attendance percentage (0–100): ", 0, 100)
internal_marks = get_valid_input("Internal marks (0–100): ", 0, 100)
sleep_hours = get_valid_input("Daily sleep hours (0–8): ", 0, 8)

# ---------- ML PROBABILITY ----------
student_input = pd.DataFrame(
    [[attendance, study_hours * 7, internal_marks]],
    columns=FEATURES
)

proba = model.predict_proba(student_input)[0]
pass_prob = round(proba[1] * 100, 1)
fail_prob = round(proba[0] * 100, 1)

# ---------- FIX 1: RULE-BASED WARNING SYSTEM ----------
# This generates warning_text AND catches edge cases ML misses
warnings = []

if attendance < 75:
    warnings.append("Low attendance")
if study_hours < 0.5:
    warnings.append("Insufficient study hours")
if internal_marks < 40:
    warnings.append("Low internal marks")
if sleep_hours < 6:
    warnings.append("Poor sleep - affects performance")

# ---------- FIX 2: FINAL DECISION (ML + RULES COMBINED) ----------
# If ML says pass BUT rules detect critical issues → still warn
if fail_prob >= 40 or len(warnings) >= 2:
    final_result = "AT RISK OF FAILING THE FINAL EXAM"
elif fail_prob >= 25 or len(warnings) == 1:
    final_result = "BORDERLINE - NEEDS IMPROVEMENT"
else:
    final_result = "LIKELY TO CLEAR THE FINAL EXAM"

# FIX 1: warning_text now properly defined
warning_text = ", ".join(warnings) if warnings else "None"

print("\n--- ML EARLY WARNING RESULT ---")
print("➡️", final_result)
print(f"   Pass Probability : {pass_prob}%")
print(f"   Fail Probability : {fail_prob}%")
if warnings:
    print(f"   ⚠️  Warnings       : {warning_text}")

# ---------- SAVE RESULT TO EXCEL ----------
unique_key = f"{reg_no}_{subject_code}"

utc_time = datetime.now(timezone.utc)
ist_time = utc_time + timedelta(hours=5, minutes=30)

result_row = pd.DataFrame([{
    "Record ID": unique_key,
    "Timestamp": ist_time.strftime("%Y-%m-%d %H:%M:%S"),
    "Register Number": reg_no,
    "Academic Year": year,
    "Subject Code": subject_code,
    "Difficulty": difficulty,
    "Attendance (%)": attendance,
    "Study Hours": study_hours,
    "Internal Marks": internal_marks,
    "Sleep Hours": sleep_hours,
    "Warnings": warning_text,
    "Pass Probability (%)": pass_prob,
    "Fail Probability (%)": fail_prob,
    "Prediction": final_result
}])

file_path = "/content/drive/MyDrive/Student_Final_Exam_Prediction_02.xlsx"

try:
    existing = pd.read_excel(file_path)
    existing = existing[existing["Record ID"] != unique_key]
    final_df = pd.concat([existing, result_row], ignore_index=True)
except FileNotFoundError:
    final_df = result_row

final_df.to_excel(file_path, index=False)

print("\n📁 RESULT STORED SUCCESSFULLY")
print(f"📄 File updated at: {file_path}")